# 04 — Model Training & Validation

> **Mục tiêu:** huấn luyện và so sánh baseline/model ứng viên trên ba bộ `CORE`, `NO_SHORTCUT`, `FULL`; dùng leave-one-simulation-run-out CV; chọn champion hoặc provisional candidate trên validation mà không mở test.
>
> **Input:** `data/processed/train.parquet`, `validation.parquet`, `split_manifest.json`, `feature_validation_report.csv`.
>
> **Output:** CV metrics, validation metrics/predictions, scenario recall, feature importance, selection pipeline và training manifest trong `models/transaction_fraud/`.

## Table of Contents

1. [Modeling contract và upstream gates](#1)
2. [Đọc dữ liệu và feature tiers](#2)
3. [Metric, weighting và pipeline factory](#3)
4. [Business/Dummy baselines](#4)
5. [Leave-one-run-out cross-validation](#5)
6. [So sánh CV](#6)
7. [Fit toàn bộ train và đánh giá validation](#7)
8. [Ablation và chọn model](#8)
9. [Scenario recall](#9)
10. [Feature importance](#10)
11. [Xuất artifacts và handoff](#11)

---
<a id="1"></a>
## 1. Modeling contract và upstream gates

Notebook dùng các train/validation run đúng theo `split_manifest.json`; test run cuối được giữ kín cho Notebook 05.

Notebook dừng ngay nếu artifact từ Notebook 03 không xác nhận: rolling strictly `< T`, future device timestamp đã bị cách ly, và registry phủ đủ FULL tier. Source generator vẫn phải được sửa trước lần tái sinh dữ liệu tiếp theo nếu audit ghi nhận `first_seen_at > T`.

In [ ]:
import json
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, brier_score_loss, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = REPO_ROOT / 'notebooks' / 'src'
if str(SRC_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC_DIR.resolve()))
from viz_utils import PALETTE, clean_ax, setup
setup()

DATA_DIR = REPO_ROOT / 'data' / 'processed'
MODEL_DIR = REPO_ROOT / 'models' / 'transaction_fraud'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
ALERT_RATE = 0.03
ALERT_RATES = (0.01, 0.02, 0.03, 0.05)
MIN_CV_PR_AUC = 0.80
MAX_CV_PR_AUC_STD = 0.10
MIN_CV_WORST_SCENARIO_RECALL = 0.80

required = [
    DATA_DIR / 'train.parquet', DATA_DIR / 'validation.parquet',
    DATA_DIR / 'split_manifest.json', DATA_DIR / 'feature_validation_report.csv',
]
missing_files = [str(p) for p in required if not p.exists()]
assert not missing_files, f'Bạn cần chạy lại notebook 03. Thiếu artifacts: {missing_files}'
assert (DATA_DIR / 'test.parquet').exists(), 'Thiếu test holdout; notebook này chỉ kiểm tra file tồn tại và không đọc nội dung.'
print('✓ Modeling contract được khởi tạo; test.parquet chưa được đọc.')

---
<a id="2"></a>
## 2. Đọc dữ liệu và feature tiers

Đọc train/validation, xác minh **không có HARD FAIL** từ Notebook 03, kiểm tra schema và entity isolation. Soft warning được hiển thị nhưng không làm notebook chết giữa demo. Test chỉ được kiểm tra tồn tại, tuyệt đối không đọc nội dung.

In [ ]:
with open(DATA_DIR / 'split_manifest.json', encoding='utf-8') as f:
    split_manifest = json.load(f)

df_train = pd.read_parquet(DATA_DIR / 'train.parquet')
df_valid = pd.read_parquet(DATA_DIR / 'validation.parquet')
validation_report = pd.read_csv(DATA_DIR / 'feature_validation_report.csv')

required_report_cols = {'Severity', 'Kết quả'}
assert required_report_cols.issubset(validation_report.columns), 'Cần chạy lại Notebook 03 phiên bản hard/soft gate mới.'
hard_failures = validation_report.loc[
    validation_report['Severity'].eq('HARD') & validation_report['Kết quả'].eq('FAIL')
]
assert hard_failures.empty, f"Notebook 03 có HARD FAIL: {hard_failures.iloc[:, 0].tolist()}"
soft_warnings = validation_report.loc[validation_report['Kết quả'].eq('WARN')]
if not soft_warnings.empty:
    print('⚠ Soft warnings từ Notebook 03:', soft_warnings.iloc[:, 0].tolist())

TARGET = split_manifest['target_column']
FEATURE_TIERS = {k: v['columns'] for k, v in split_manifest['feature_subsets'].items()}
FEATURE_TIERS = {'CORE': FEATURE_TIERS['CORE_FEATURES'], 'NO_SHORTCUT': FEATURE_TIERS['NO_SHORTCUT_FEATURES'], 'FULL': FEATURE_TIERS['FULL_FEATURES']}
AUDIT_REQUIRED = ['transaction_id', 'account_id', 'customer_id', 'simulation_run_id', 'event_id', 'scenario_code', 'label_scope', 'sample_weight', 'hard_negative', 'amount_num', 'beneficiary_id', 'txn_time_utc']

missing_columns = {tier: sorted(set(cols) - set(df_train.columns)) for tier, cols in FEATURE_TIERS.items()}
assert not any(missing_columns.values()), f'Feature thiếu trong train: {missing_columns}'
assert not (set(AUDIT_REQUIRED) - set(df_train.columns)), 'Train thiếu audit columns.'
assert set(df_train['simulation_run_id'].unique()) == set(split_manifest['train_set']['runs'])
assert set(df_valid['simulation_run_id'].unique()) == set(split_manifest['validation_set']['runs'])
assert set(df_train['account_id']).isdisjoint(df_valid['account_id'])
assert set(df_train['customer_id']).isdisjoint(df_valid['customer_id'])

zero_variance = {tier: [c for c in cols if df_train[c].nunique(dropna=False) <= 1] for tier, cols in FEATURE_TIERS.items()}
if any(zero_variance.values()):
    print('⚠ Zero-variance features không chặn research:', {k: v for k, v in zero_variance.items() if v})

registry = pd.read_csv(DATA_DIR / 'feature_registry.csv')
registry_gap = sorted(set(FEATURE_TIERS['FULL']) - set(registry['feature_name']))
assert not registry_gap, f'Feature registry chưa đầy đủ: {registry_gap}'
print(f'✓ Train: {len(df_train):,} | Validation: {len(df_valid):,}')
print({tier: len(cols) for tier, cols in FEATURE_TIERS.items()})
print(f'✓ Registry phủ đủ {len(FEATURE_TIERS["FULL"])} FULL features.')

---
<a id="3"></a>
## 3. Metric, weighting và pipeline factory

`sample_weight` cân bằng đóng góp giữa các event nhiều dòng. Class balancing chỉ áp dụng cho model học có giám sát; Dummy giữ prior fraud gốc. Metric chính là event-weighted PR-AUC, kèm recall theo capacity, hard-negative FPR và worst-scenario recall.

`brier_score_uncalibrated` chỉ mô tả score thô sau class weighting, chưa được diễn giải là xác suất fraud thực. Calibration và threshold production thuộc Notebook 05.

In [ ]:
CATEGORICAL_CANDIDATES = [
    'channel', 'customer_segment', 'kyc_level', 'base_risk_level',
    'account_type', 'device_type', 'os',
]

def make_preprocessor(features, scale_numeric=True):
    categorical = [c for c in CATEGORICAL_CANDIDATES if c in features]
    numeric = [c for c in features if c not in categorical]
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        numeric_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(numeric_steps), numeric),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='UNKNOWN')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), categorical),
    ], remainder='drop')

def balanced_event_weights(y, event_weight):
    y_arr = np.asarray(y, dtype=int)
    w = np.asarray(event_weight, dtype=float)
    totals = {cls: w[y_arr == cls].sum() for cls in (0, 1)}
    assert min(totals.values()) > 0, f'Fold thiếu class: {totals}'
    total = sum(totals.values())
    factors = {cls: total / (2.0 * totals[cls]) for cls in (0, 1)}
    return w * np.where(y_arr == 1, factors[1], factors[0])

def model_fit_weights(kind, frame):
    event_weight = frame['sample_weight'].astype(float).to_numpy()
    if kind == 'dummy':
        return event_weight
    return balanced_event_weights(frame[TARGET], event_weight)

def top_rate_predictions(scores, alert_rate=ALERT_RATE):
    scores = np.asarray(scores, dtype=float)
    k = max(1, int(np.ceil(len(scores) * alert_rate)))
    chosen = np.argsort(-scores, kind='mergesort')[:k]
    pred = np.zeros(len(scores), dtype=int)
    pred[chosen] = 1
    return pred, float(scores[chosen[-1]])

def scenario_recall(frame, pred):
    positive_mask = frame[TARGET].eq(1).to_numpy()
    tmp = frame.loc[positive_mask, ['scenario_code', 'sample_weight']].copy()
    tmp['prediction'] = np.asarray(pred)[positive_mask]
    rows = []
    for scenario, group in tmp.groupby('scenario_code'):
        rows.append({
            'scenario_code': scenario,
            'positive_transactions': len(group),
            'recall_txn': group['prediction'].mean(),
            'recall_event_weighted': np.average(group['prediction'], weights=group['sample_weight']),
        })
    return pd.DataFrame(rows).sort_values('scenario_code').reset_index(drop=True)

def metric_record(frame, scores, experiment, tier, split_name, alert_rate=ALERT_RATE):
    y = frame[TARGET].astype(int).to_numpy()
    event_w = frame['sample_weight'].astype(float).to_numpy()
    pred, threshold = top_rate_predictions(scores, alert_rate)
    hn = frame['hard_negative'].astype(int).to_numpy() == 1
    scen = scenario_recall(frame, pred)
    return {
        'experiment': experiment, 'tier': tier, 'split': split_name, 'alert_rate': alert_rate,
        'pr_auc_event_weighted': average_precision_score(y, scores, sample_weight=event_w),
        'pr_auc_txn': average_precision_score(y, scores),
        'roc_auc': roc_auc_score(y, scores),
        'brier_score_uncalibrated': brier_score_loss(y, scores, sample_weight=event_w),
        'precision_at_alert_rate': precision_score(y, pred, zero_division=0),
        'recall_at_alert_rate': recall_score(y, pred, zero_division=0),
        'f1_at_alert_rate': f1_score(y, pred, zero_division=0),
        'hard_negative_fpr': float(pred[hn].mean()) if hn.any() else np.nan,
        'worst_scenario_recall': float(scen['recall_event_weighted'].min()),
        'threshold_at_alert_rate': threshold,
        'alerts_per_1000': pred.mean() * 1000.0,
    }

def business_rule_operational_score(frame):
    # Baseline nghiệp vụ rộng, có composite shortcut signals.
    return (
        2.0 * frame['is_rapid_new_beneficiary'].astype(float)
        + 2.0 * frame['is_ato_sequence'].astype(float)
        + frame['is_high_balance_drain'].astype(float)
        + (frame['prior_txn_count_10m'] >= 2).astype(float)
        + (frame['amount_to_limit_ratio'] >= 0.8).astype(float)
    ).to_numpy() / 7.0

def business_rule_core_score(frame):
    # Baseline công bằng với CORE: không dùng composite shortcut candidates.
    return (
        frame['is_high_balance_drain'].astype(float)
        + (frame['prior_txn_count_10m'] >= 2).astype(float)
        + (frame['amount_to_limit_ratio'] >= 0.8).astype(float)
    ).to_numpy() / 3.0

def make_pipeline(features, estimator, scale_numeric=True):
    return Pipeline([
        ('preprocessor', make_preprocessor(features, scale_numeric=scale_numeric)),
        ('model', estimator),
    ])

---
<a id="4"></a>
## 4. Business/Dummy baselines

- `business_rule_operational`: rule rộng, có composite signals thuộc shortcut candidates.
- `business_rule_core`: rule công bằng với CORE, bỏ hai composite signals.
- Dummy học fraud prior từ event weight gốc, không class-balance.

Hai business baseline được báo riêng, không trộn phạm vi feature khi diễn giải.

In [ ]:
EXPERIMENTS = [
    {'name': 'dummy_core', 'tier': 'CORE', 'kind': 'dummy'},
    {'name': 'logistic_core', 'tier': 'CORE', 'kind': 'logistic'},
    {'name': 'logistic_no_shortcut', 'tier': 'NO_SHORTCUT', 'kind': 'logistic'},
    {'name': 'logistic_full', 'tier': 'FULL', 'kind': 'logistic'},
    {'name': 'tree_no_shortcut', 'tier': 'NO_SHORTCUT', 'kind': 'tree'},
    {'name': 'histgb_no_shortcut', 'tier': 'NO_SHORTCUT', 'kind': 'histgb'},
    {'name': 'histgb_full', 'tier': 'FULL', 'kind': 'histgb'},
]

def estimator_for(kind):
    if kind == 'dummy':
        return DummyClassifier(strategy='prior'), True
    if kind == 'logistic':
        return LogisticRegression(C=1.0, max_iter=800, solver='lbfgs', random_state=RANDOM_STATE), True
    if kind == 'tree':
        return DecisionTreeClassifier(max_depth=7, min_samples_leaf=50, random_state=RANDOM_STATE), False
    if kind == 'histgb':
        return HistGradientBoostingClassifier(
            learning_rate=0.08, max_iter=160, max_leaf_nodes=31,
            min_samples_leaf=30, l2_regularization=1.0,
            early_stopping=True, random_state=RANDOM_STATE,
        ), False
    raise ValueError(f'Estimator chưa hỗ trợ: {kind}')

operational_rule_scores = business_rule_operational_score(df_valid)
core_rule_scores = business_rule_core_score(df_valid)
baseline_metrics = [
    metric_record(df_valid, operational_rule_scores, 'business_rule_operational', 'RULE_OPERATIONAL', 'validation'),
    metric_record(df_valid, core_rule_scores, 'business_rule_core', 'RULE_CORE', 'validation'),
]
display(pd.DataFrame(baseline_metrics))

---
<a id="5"></a>
## 5. Leave-one-run-out cross-validation

Mỗi fold giữ lại một simulation run trong ba train runs. Toàn bộ preprocessing được fit lại bên trong fold và trọng số class chỉ được tính từ phần fit của fold.


In [ ]:
cv_rows = []
train_runs = sorted(df_train['simulation_run_id'].unique())
t0 = time.time()

for spec in EXPERIMENTS:
    features = FEATURE_TIERS[spec['tier']]
    for held_run in train_runs:
        fit_df = df_train[df_train['simulation_run_id'] != held_run]
        fold_df = df_train[df_train['simulation_run_id'] == held_run]
        estimator, scale_numeric = estimator_for(spec['kind'])
        pipeline = make_pipeline(features, estimator, scale_numeric)
        fit_weight = model_fit_weights(spec['kind'], fit_df)
        pipeline.fit(fit_df[features], fit_df[TARGET], model__sample_weight=fit_weight)
        fold_scores = pipeline.predict_proba(fold_df[features])[:, 1]
        row = metric_record(fold_df, fold_scores, spec['name'], spec['tier'], held_run)
        cv_rows.append(row)
    print(f"✓ CV xong: {spec['name']}")

cv_fold_metrics = pd.DataFrame(cv_rows)
cv_summary = (
    cv_fold_metrics.groupby(['experiment', 'tier'], as_index=False)
    .agg(
        cv_pr_auc_mean=('pr_auc_event_weighted', 'mean'),
        cv_pr_auc_std=('pr_auc_event_weighted', 'std'),
        cv_recall_mean=('recall_at_alert_rate', 'mean'),
        cv_hn_fpr_mean=('hard_negative_fpr', 'mean'),
        cv_worst_scenario_recall=('worst_scenario_recall', 'min'),
    )
    .sort_values(['cv_pr_auc_mean', 'cv_hn_fpr_mean'], ascending=[False, True])
    .reset_index(drop=True)
)
print(f'Hoàn thành {len(cv_rows)} CV fits trong {time.time() - t0:.1f}s.')
display(cv_summary)

---
<a id="6"></a>
## 6. So sánh CV

Dashboard so sánh chất lượng trung bình và độ dao động qua ba simulation run. Model tốt cần PR-AUC cao nhưng không đánh đổi bằng hard-negative FPR hoặc bỏ sót toàn bộ một scenario.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
order = cv_summary.sort_values('cv_pr_auc_mean', ascending=True)['experiment']
sns.barplot(data=cv_summary, y='experiment', x='cv_pr_auc_mean', order=order, color=PALETTE['primary'], ax=axes[0])
clean_ax(axes[0], 'A. Event-weighted PR-AUC qua 3 CV folds', 'Mean PR-AUC', 'Experiment',
         'Càng cao càng tốt; error bar được đọc từ bảng cv_pr_auc_std')
sns.scatterplot(data=cv_summary, x='cv_hn_fpr_mean', y='cv_recall_mean', hue='tier', s=110, ax=axes[1])
clean_ax(axes[1], 'B. Recall và Hard-negative FPR tại 30 alerts/1.000', 'Hard-negative FPR', 'Recall',
         'Ưu tiên góc trên-trái')
plt.tight_layout()
plt.show()


---
<a id="7"></a>
## 7. Fit toàn bộ train và đánh giá validation

Fit lại từng ứng viên trên toàn bộ train runs rồi đánh giá trên validation run do manifest chỉ định. Dummy dùng event weight gốc; các supervised model mới dùng class-balanced event weight. Brier ở đây là score chưa calibration, không phải xác suất production.

In [ ]:
validation_rows = list(baseline_metrics)
fitted_models = {}
validation_scores = {
    'business_rule_operational': operational_rule_scores,
    'business_rule_core': core_rule_scores,
}
t0 = time.time()

for spec in EXPERIMENTS:
    features = FEATURE_TIERS[spec['tier']]
    estimator, scale_numeric = estimator_for(spec['kind'])
    pipeline = make_pipeline(features, estimator, scale_numeric)
    started = time.time()
    fit_weight = model_fit_weights(spec['kind'], df_train)
    pipeline.fit(df_train[features], df_train[TARGET], model__sample_weight=fit_weight)
    scores = pipeline.predict_proba(df_valid[features])[:, 1]
    row = metric_record(df_valid, scores, spec['name'], spec['tier'], split_manifest['validation_set']['runs'][0])
    row['fit_seconds'] = time.time() - started
    validation_rows.append(row)
    fitted_models[spec['name']] = pipeline
    validation_scores[spec['name']] = scores
    print(f"✓ Validation xong: {spec['name']}")

validation_metrics = (
    pd.DataFrame(validation_rows)
    .sort_values(['pr_auc_event_weighted', 'hard_negative_fpr'], ascending=[False, True])
    .reset_index(drop=True)
)
print(f'Hoàn thành validation trong {time.time() - t0:.1f}s.')
display(validation_metrics)

---
<a id="8"></a>
## 8. Ablation và chọn model

Ứng viên đạt đủ CV gate mới được gọi là `CHAMPION`. Nếu không ứng viên nào đạt, notebook vẫn hoàn thành để phục vụ demo nhưng chỉ tạo `PROVISIONAL_RELAXED_CV_GATE`; trạng thái này được ghi vào manifest và không được xuất dưới tên `champion_pipeline.joblib`.

Trong nhóm hợp lệ, ưu tiên tier ít shortcut rồi mới xét hard-negative FPR, CV stability và PR-AUC. Capacity 10/20/30/50 alerts trên 1.000 vẫn chỉ là policy comparison, chưa phải threshold production.

In [ ]:
baseline_names = {'business_rule_operational', 'business_rule_core', 'dummy_core'}
candidate_metrics = (
    validation_metrics.loc[~validation_metrics['experiment'].isin(baseline_names)]
    .merge(cv_summary, on=['experiment', 'tier'], how='inner', validate='one_to_one')
)
candidate_metrics['cv_gate_pass'] = (
    candidate_metrics['cv_pr_auc_mean'].ge(MIN_CV_PR_AUC)
    & candidate_metrics['cv_pr_auc_std'].le(MAX_CV_PR_AUC_STD)
    & candidate_metrics['cv_worst_scenario_recall'].ge(MIN_CV_WORST_SCENARIO_RECALL)
)

champion_selected_under_relaxed_gate = not candidate_metrics['cv_gate_pass'].any()
selection_status = 'PROVISIONAL_RELAXED_CV_GATE' if champion_selected_under_relaxed_gate else 'CHAMPION'
selection_pool = candidate_metrics.copy() if champion_selected_under_relaxed_gate else candidate_metrics.loc[candidate_metrics['cv_gate_pass']].copy()
if champion_selected_under_relaxed_gate:
    print('⚠ Không model nào vượt đủ CV gate; chỉ chọn provisional candidate để notebook tiếp tục.')

best_validation_pr = selection_pool['pr_auc_event_weighted'].max()
selection_pool = selection_pool.loc[selection_pool['pr_auc_event_weighted'] >= 0.99 * best_validation_pr].copy()
tier_priority = {'CORE': 0, 'NO_SHORTCUT': 1, 'FULL': 2}
selection_pool['tier_priority'] = selection_pool['tier'].map(tier_priority)
selection_pool = selection_pool.sort_values(
    ['tier_priority', 'hard_negative_fpr', 'cv_pr_auc_std', 'cv_worst_scenario_recall', 'pr_auc_event_weighted'],
    ascending=[True, True, True, False, False],
)

selected_name = selection_pool.iloc[0]['experiment']
selected_tier = selection_pool.iloc[0]['tier']
selected_pipeline = fitted_models[selected_name]
selected_scores = validation_scores[selected_name]
selected_pred, selected_threshold = top_rate_predictions(selected_scores)

alert_capacity = pd.DataFrame([
    metric_record(df_valid, selected_scores, selected_name, selected_tier, 'validation', rate)
    for rate in ALERT_RATES
])
display(candidate_metrics.sort_values(['cv_gate_pass', 'cv_pr_auc_mean'], ascending=[False, False]))
display(alert_capacity[['alerts_per_1000', 'threshold_at_alert_rate', 'precision_at_alert_rate', 'recall_at_alert_rate', 'hard_negative_fpr', 'worst_scenario_recall']])

ablation_view = validation_metrics[validation_metrics['experiment'].str.contains('logistic|histgb', regex=True)].copy()
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.scatterplot(data=ablation_view, x='hard_negative_fpr', y='pr_auc_event_weighted', hue='tier', style='experiment', s=140, ax=ax)
clean_ax(ax, 'Ablation: chất lượng và phụ thuộc shortcut', 'Hard-negative FPR', 'Event-weighted PR-AUC', 'CORE / NO_SHORTCUT / FULL trên validation')
plt.tight_layout()
plt.show()
print(f'{selection_status}: {selected_name} | tier={selected_tier} | top-{ALERT_RATE:.0%}={selected_threshold:.6f}')

---
<a id="9"></a>
## 9. Scenario recall

Báo cáo recall riêng cho từng fraud scenario ở validation. Event-weighted recall ngăn các scenario nhiều transaction như TXN-04/TXN-07 chi phối kết luận.


In [ ]:
selected_scenario_recall = scenario_recall(df_valid, selected_pred)
display(selected_scenario_recall)

fig, ax = plt.subplots(figsize=(12, 5.5))
sns.barplot(data=selected_scenario_recall, x='scenario_code', y='recall_event_weighted',
            color=PALETTE['danger'], ax=ax)
ax.set_ylim(0, 1.05)
clean_ax(ax, 'Validation recall theo fraud scenario', 'Scenario', 'Event-weighted recall',
         f'{selected_name} tại {ALERT_RATE * 1000:.0f} alerts/1.000 giao dịch')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

---
<a id="10"></a>
## 10. Feature importance

Permutation importance được tính trên một mẫu validation có giữ toàn bộ fraud. Cách này đo đóng góp của feature raw qua toàn pipeline và áp dụng thống nhất cho Logistic, Tree và HistGradientBoosting.


In [ ]:
selected_features = FEATURE_TIERS[selected_tier]
fraud_idx = df_valid.index[df_valid[TARGET].eq(1)]
normal_idx = df_valid.index[df_valid[TARGET].eq(0)]
normal_n = min(5000 - len(fraud_idx), len(normal_idx))
sample_idx = fraud_idx.union(df_valid.loc[normal_idx].sample(normal_n, random_state=RANDOM_STATE).index)
importance_frame = df_valid.loc[sample_idx]

def validation_ap_scorer(estimator, X, y):
    return average_precision_score(y, estimator.predict_proba(X)[:, 1])

perm = permutation_importance(
    selected_pipeline, importance_frame[selected_features], importance_frame[TARGET],
    scoring=validation_ap_scorer, n_repeats=2, random_state=RANDOM_STATE, n_jobs=1,
)
feature_importance = (
    pd.DataFrame({
        'feature': selected_features,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    })
    .sort_values('importance_mean', ascending=False)
    .reset_index(drop=True)
)
display(feature_importance.head(20))

fig, ax = plt.subplots(figsize=(10, 7))
top_imp = feature_importance.head(15).sort_values('importance_mean')
ax.barh(top_imp['feature'], top_imp['importance_mean'], color=PALETTE['primary'])
clean_ax(ax, 'Top 15 permutation importance trên validation', 'Giảm PR-AUC khi hoán vị', 'Feature',
         'Giá trị gần 0 cho thấy feature ít đóng góp thêm trong selected')
plt.tight_layout()
plt.show()

---
<a id="11"></a>
## 11. Xuất artifacts và handoff

Lưu selection pipeline, trạng thái CV gate, metric validation, bảng capacity và predictions. Chỉ selection đạt `CHAMPION` mới có deployment artifact. Test holdout vẫn chưa được đọc.

In [ ]:
cv_fold_metrics.to_csv(MODEL_DIR / 'cv_fold_metrics.csv', index=False)
cv_summary.to_csv(MODEL_DIR / 'cv_summary.csv', index=False)
validation_metrics.to_csv(MODEL_DIR / 'validation_metrics.csv', index=False)
candidate_metrics.to_csv(MODEL_DIR / 'champion_candidates.csv', index=False)
alert_capacity.to_csv(MODEL_DIR / 'validation_alert_capacity.csv', index=False)
selected_scenario_recall.to_csv(MODEL_DIR / 'validation_scenario_recall.csv', index=False)
feature_importance.to_csv(MODEL_DIR / 'feature_importance.csv', index=False)

prediction_cols = [c for c in AUDIT_REQUIRED if c in df_valid.columns] + [TARGET]
validation_predictions = df_valid[prediction_cols].copy()
validation_predictions['fraud_score_uncalibrated'] = selected_scores
validation_predictions['prediction_at_alert_rate'] = selected_pred
validation_predictions.to_parquet(MODEL_DIR / 'validation_predictions.parquet', index=False)

selection_artifact = MODEL_DIR / 'selected_pipeline.joblib'
champion_artifact = MODEL_DIR / 'champion_pipeline.joblib'
joblib.dump(selected_pipeline, selection_artifact)
if champion_selected_under_relaxed_gate:
    champion_artifact.unlink(missing_ok=True)  # Không để champion cũ gây hiểu nhầm.
    deployment_artifact = None
else:
    joblib.dump(selected_pipeline, champion_artifact)
    deployment_artifact = champion_artifact.name

processed_feature_names = selected_pipeline.named_steps['preprocessor'].get_feature_names_out()
pd.DataFrame({'processed_feature_name': processed_feature_names}).to_csv(MODEL_DIR / 'processed_feature_names.csv', index=False)
selected_metrics = validation_metrics.loc[validation_metrics['experiment'].eq(selected_name)].iloc[0].to_dict()
training_manifest = {
    'random_state': RANDOM_STATE,
    'selection_status': selection_status,
    'champion_selected_under_relaxed_gate': champion_selected_under_relaxed_gate,
    'selected_experiment': selected_name,
    'selected_tier': selected_tier,
    'selected_pipeline_artifact': selection_artifact.name,
    'deployment_artifact': deployment_artifact,
    'selected_raw_feature_count': len(selected_features),
    'selected_processed_feature_count': len(processed_feature_names),
    'train_runs': sorted(df_train['simulation_run_id'].unique().tolist()),
    'validation_runs': sorted(df_valid['simulation_run_id'].unique().tolist()),
    'test_opened': False,
    'selection_rule': 'Pass CV gates; within 1% validation PR-AUC prefer lower-shortcut tier, lower HN FPR and lower CV std.',
    'cv_gates': {'min_pr_auc_mean': MIN_CV_PR_AUC, 'max_pr_auc_std': MAX_CV_PR_AUC_STD, 'min_worst_scenario_recall': MIN_CV_WORST_SCENARIO_RECALL},
    'alert_rates_evaluated': list(ALERT_RATES),
    'validation_metrics': {k: (v.item() if hasattr(v, 'item') else v) for k, v in selected_metrics.items()},
    'upstream_limitations': [
        f"Raw future device first_seen rows quarantined: {split_manifest.get('data_quality', {}).get('future_device_first_seen_rows_quarantined', 0)}.",
        'Class-weighted scores require calibration in Notebook 05.',
        'Synthetic separation remains subject to shortcut/generalization audit.',
    ],
}
with open(MODEL_DIR / 'training_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(training_manifest, f, indent=2, ensure_ascii=False)

print(f'✓ Selection artifact: {selection_artifact.name}')
print(f'✓ Status: {selection_status}; deployment artifact: {deployment_artifact}')
print('✓ Test holdout chưa được mở.')

### Handoff sang Notebook 05

Chỉ khi `training_manifest.json.selection_status == "CHAMPION"` và `deployment_artifact` khác null, Notebook 05 mới được nạp champion để calibration/test evaluation. Trạng thái `PROVISIONAL_RELAXED_CV_GATE` chỉ phục vụ chẩn đoán/demo và không đủ điều kiện mở test hoặc triển khai SAS.